# The Real-World Clustering Pipeline — Start to End

How a practitioner actually does this. No hand-coded algorithms, no pointless 2D plots. PCA by a **variance threshold**, clustering on the full data, evaluation by **numbers**, and plots only where they earn their place.

**Dataset:** Breast Cancer Wisconsin — 569 tumor samples, **30 features** (cell-nucleus measurements), 2 real classes (malignant / benign). 30 dimensions is genuinely un-eyeball-able, so this behaves like real work.

**The pipeline:**
```
1. Load & inspect
2. Standardize          (always)
3. PCA by variance      (keep 95%, not a magic '2')
4. Choose k             (silhouette across k)
5. Cluster              (KMeans / GMM / DBSCAN)
6. Evaluate             (4 metrics + confusion matrix)
7. Sanity-glance plot   (optional, after the fact)
```
---

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             adjusted_rand_score, confusion_matrix,
                             davies_bouldin_score, calinski_harabasz_score)

np.random.seed(0)

## Step 1 — Load and inspect

First thing in real work: understand the shape and the scales. `y` (the diagnosis) is set aside — we pretend we don't have it, and only use it at the very end to check our work. This mimics reality, where you cluster precisely *because* you don't have labels.

In [2]:
data = load_breast_cancer()
X = data.data
y_true = data.target            # 0=malignant, 1=benign — HELD OUT, used only to score at the end
feature_names = data.feature_names

df = pd.DataFrame(X, columns=feature_names)
print(f"{X.shape[0]} samples, {X.shape[1]} features")
print(f"\nFeature scales are all over the place:")
print(df[['mean radius', 'mean area', 'mean smoothness', 'worst area']].describe().loc[['mean','min','max']].round(2))
print("\n'mean area' is in the hundreds; 'mean smoothness' is ~0.1.")
print("Distance-based clustering would be hijacked by 'area'. -> must standardize.")

569 samples, 30 features

Feature scales are all over the place:
      mean radius  mean area  mean smoothness  worst area
mean        14.13     654.89             0.10      880.58
min          6.98     143.50             0.05      185.20
max         28.11    2501.00             0.16     4254.00

'mean area' is in the hundreds; 'mean smoothness' is ~0.1.
Distance-based clustering would be hijacked by 'area'. -> must standardize.


## Step 2 — Standardize (always)

Non-negotiable when features have different units. Puts everything on mean 0, std 1 so no feature dominates the distance math by virtue of its scale.

In [3]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
print("After standardizing, every feature is mean~0 std~1.")
print("Spot check 'mean area':  mean =", round(X_std[:,3].mean(), 4), " std =", round(X_std[:,3].std(), 4))

After standardizing, every feature is mean~0 std~1.
Spot check 'mean area':  mean = -0.0  std = 1.0


## Step 3 — PCA by variance threshold (the real way)

Not `PCA(n_components=2)` — that's a tutorial crutch. In real work you say "keep enough components to retain 95% of the variance" and let PCA decide how many that is. Here we check how the variance accumulates, then keep 95%.

In [4]:
# Fit PCA on everything first, just to see the variance curve
pca_full = PCA().fit(X_std)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, len(cum_var)+1)), y=cum_var,
    mode='lines+markers', marker_color='#534AB7', name='cumulative variance'))
fig.add_hline(y=0.95, line_dash='dash', line_color='red',
    annotation_text='95% threshold')
fig.update_layout(title='How many components to keep 95% of the information?',
    xaxis_title='number of components', yaxis_title='cumulative variance kept',
    width=700, height=420)
fig.show()

# Now keep 95% — PCA picks the count automatically
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_std)
print(f"PCA kept {X_pca.shape[1]} components (out of {X.shape[1]}) to retain 95% variance")
print("This is the matrix we cluster on. Note: it's 10-D, NOT 2-D. We never plot it directly.")

PCA kept 10 components (out of 30) to retain 95% variance
This is the matrix we cluster on. Note: it's 10-D, NOT 2-D. We never plot it directly.


**This is the key real-world move.** We went from 30 features to 10 components, keeping 95% of the information. That 10-D matrix is what we cluster — we will never make a "2D PCA plot" to do the clustering. (We do have a quick numbers check below on whether PCA even helps here — sometimes it does, sometimes it doesn't.)

## Quick check: does PCA actually help here? (numbers, not plots)

In [5]:
# Compare clustering on full standardized data vs PCA-95% data
for name, data_matrix in [('standardized (30-D)', X_std), ('PCA 95% (10-D)', X_pca)]:
    km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(data_matrix)
    sil = silhouette_score(data_matrix, km.labels_)
    ari = adjusted_rand_score(y_true, km.labels_)
    print(f"{name:22s}  silhouette={sil:.3f}  ARI={ari:.3f}")
print("\nHere PCA helps a little (drops noisy directions). On other data it's a wash.")
print("Either is fine — we'll proceed on the PCA-95% matrix since it's smaller and slightly better.")
X_work = X_pca   # the matrix we'll use going forward

standardized (30-D)     silhouette=0.343  ARI=0.654
PCA 95% (10-D)          silhouette=0.356  ARI=0.654

Here PCA helps a little (drops noisy directions). On other data it's a wash.
Either is fine — we'll proceed on the PCA-95% matrix since it's smaller and slightly better.


## Step 4 — Choose k by silhouette (no labels needed)

In real life you don't know how many clusters exist. Sweep k, compute the silhouette for each, pick the peak. (We secretly know it's 2 — good confirmation.)

In [6]:
ks = range(2, 9)
sils, inertias = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_work)
    sils.append(silhouette_score(X_work, km.labels_))
    inertias.append(km.inertia_)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Silhouette (pick the peak)', 'Elbow (find the bend)'))
fig.add_trace(go.Scatter(x=list(ks), y=sils, mode='lines+markers', marker_color='#1D9E75'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(ks), y=inertias, mode='lines+markers', marker_color='#534AB7'), row=1, col=2)
fig.update_xaxes(title_text='k', row=1, col=1); fig.update_xaxes(title_text='k', row=1, col=2)
fig.update_yaxes(title_text='silhouette', row=1, col=1); fig.update_yaxes(title_text='inertia', row=1, col=2)
fig.update_layout(width=900, height=400, showlegend=False,
    title='Choosing k with no labels')
fig.show()

best_k = list(ks)[int(np.argmax(sils))]
print(f"Best k by silhouette: {best_k}")
for k, s in zip(ks, sils): print(f"  k={k}: silhouette={s:.3f}")

Best k by silhouette: 2
  k=2: silhouette=0.356
  k=3: silhouette=0.326
  k=4: silhouette=0.297
  k=5: silhouette=0.173
  k=6: silhouette=0.179
  k=7: silhouette=0.154
  k=8: silhouette=0.160


## Step 5 — Run the final clustering

In [ ]:
final = KMeans(n_clusters=best_k, n_init=10, random_state=0).fit(X_work)
labels = final.labels_
print(labels)
print(f"Clustered into {best_k} groups.")
print(f"Cluster sizes: {np.bincount(labels)}")

[1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 0 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 1 0 0 1 1 0 0 0 0 1 0 1 1 0 0 1 0 1 0 1 0
 0 1 0 1 1 0 0 1 1 1 0 1 0 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0
 0 1 0 0 0 0 1 1 0 0 1 1 0 0 0 0 1 1 1 0 1 1 0 1 0 0 0 1 0 0 1 0 0 0 0 1 0
 0 0 0 0 1 0 0 0 1 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 1 0 0 0 1 1 0 0 1 1 0 0 0
 0 0 0 0 0 1 0 0 1 1 0 1 1 1 1 0 1 1 1 0 0 0 0 0 0 1 0 1 1 1 1 0 0 1 1 0 0
 0 1 0 0 0 0 0 1 1 0 0 1 0 0 1 1 0 1 0 0 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 1 1
 1 1 0 1 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 0 1 1 1 0 0
 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1
 1 0 1 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0
 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 1 0 1 0 0 0 0 1 

## Step 6 — Evaluation (the numbers that matter)

Two families of metric. **Internal** (no labels — what you use in real life) and **external** (needs labels — only for testing/learning).

**Internal metrics** — judge cluster shape:
- **Silhouette** (−1 → +1): how tight & separated. Higher better. The go-to.
- **Davies-Bouldin** (0 → ∞): avg similarity between each cluster and its most-similar neighbor. **Lower better** (0 = perfectly separated).
- **Calinski-Harabasz** (0 → ∞): ratio of between-cluster to within-cluster spread. **Higher better.**

**External metrics** — compare to ground truth:
- **ARI** (~0 → 1): agreement with true labels, chance-adjusted. 1 = perfect.
- **Confusion matrix**: which true class landed in which cluster.

In [8]:
print("INTERNAL (no labels — real-life metrics):")
print(f"  Silhouette        : {silhouette_score(X_work, labels):.3f}   (higher better, >0.5 strong)")
print(f"  Davies-Bouldin    : {davies_bouldin_score(X_work, labels):.3f}   (LOWER better, ~0 ideal)")
print(f"  Calinski-Harabasz : {calinski_harabasz_score(X_work, labels):.1f}  (HIGHER better)")
print()
print("EXTERNAL (needs labels — only because this is a teaching set):")
print(f"  ARI               : {adjusted_rand_score(y_true, labels):.3f}   (1=perfect, 0=random)")

INTERNAL (no labels — real-life metrics):
  Silhouette        : 0.356   (higher better, >0.5 strong)
  Davies-Bouldin    : 1.268   (LOWER better, ~0 ideal)
  Calinski-Harabasz : 288.1  (HIGHER better)

EXTERNAL (needs labels — only because this is a teaching set):
  ARI               : 0.654   (1=perfect, 0=random)


In [9]:
# Confusion matrix — the most honest 'did it work' check when you have labels
cm = confusion_matrix(y_true, labels)
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='cluster found', y='true diagnosis', color='count'),
    x=[f'cluster {i}' for i in range(best_k)], y=['malignant', 'benign'])
fig.update_layout(title='Confusion matrix: each true class should pile into one cluster',
    width=520, height=420)
fig.show()

# turn it into a readable accuracy (after matching clusters to classes)
from scipy.optimize import linear_sum_assignment
row, col = linear_sum_assignment(-cm)
matched = cm[row, col].sum()
print(f"Best cluster-to-class matching recovers {matched}/{len(y_true)} = {matched/len(y_true):.1%} of samples")
print("(Cluster numbers are arbitrary; we match them to classes to read accuracy.)")

Best cluster-to-class matching recovers 515/569 = 90.5% of samples
(Cluster numbers are arbitrary; we match them to classes to read accuracy.)


## Step 7 — The optional sanity-glance plot

Now — and only now — a 2D plot. NOT to do the clustering (that's done, on 10-D data). Just to *look* at the result and sanity-check it, and to have something to show a human. We project the 10-D data to 2D purely for display and color by the clusters we already found.

In [ ]:
# 2D projection FOR DISPLAY ONLY (clustering was on 10-D) # this only for visualization the points are at 10d only, we took only pc 1 -2  for the ploting
X_display = PCA(n_components=2).fit_transform(X_std)
colors = ['#534AB7', '#D85A30', '#1D9E75', '#D4537E']

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Our clusters (computed on 10-D)', 'True diagnosis (held out)'))
for j in range(best_k):
    p = X_display[labels == j]
    fig.add_trace(go.Scatter(x=p[:,0], y=p[:,1], mode='markers',
        marker=dict(size=5, color=colors[j], opacity=0.6), showlegend=False), row=1, col=1)
for j, nm in enumerate(['malignant', 'benign']):
    p = X_display[y_true == j]
    fig.add_trace(go.Scatter(x=p[:,0], y=p[:,1], mode='markers',
        marker=dict(size=5, color=colors[j], opacity=0.6), showlegend=False), row=1, col=2)
fig.update_layout(title='Sanity glance: clusters (left) line up with truth (right)',
    width=900, height=430)
fig.show()
print("This plot is decoration + diagnosis, NOT the engine. The clustering used 10-D data.")

This plot is decoration + diagnosis, NOT the engine. The clustering used 10-D data.


## Bonus — comparing 3 algorithms the real way (one table)

In practice you'd try a few methods and compare metrics, then pick the winner for your data.

In [11]:
candidates = {
    'KMeans(2)':  KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_work).labels_,
    'GMM(2)':     GaussianMixture(n_components=2, random_state=0).fit(X_work).predict(X_work),
    'DBSCAN':     DBSCAN(eps=3.0, min_samples=5).fit(X_work).labels_,
}

rows = []
for name, lab in candidates.items():
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    noise = int((lab == -1).sum())
    if nc > 1:
        m = lab != -1
        sil = round(silhouette_score(X_work[m], lab[m]), 3)
        db = round(davies_bouldin_score(X_work[m], lab[m]), 3)
    else:
        sil = db = 'n/a'
    ari = round(adjusted_rand_score(y_true, lab), 3)
    rows.append([name, nc, noise, sil, db, ari])

comp = pd.DataFrame(rows, columns=['method', 'clusters', 'noise', 'silhouette↑', 'davies-bouldin↓', 'ARI↑'])
print(comp.to_string(index=False))
print("\nKMeans & GMM both recover the 2 groups well. DBSCAN is eps-sensitive on this")
print("high-D data and tends to dump everything into one blob + noise — wrong tool here.")

   method  clusters  noise silhouette↑ davies-bouldin↓  ARI↑
KMeans(2)         2      0       0.356           1.268 0.654
   GMM(2)         2      0       0.263           2.187 0.137
   DBSCAN         1     83         n/a             n/a 0.068

KMeans & GMM both recover the 2 groups well. DBSCAN is eps-sensitive on this
high-D data and tends to dump everything into one blob + noise — wrong tool here.


---

# The complete real-world recipe (your reference)

```
1. INSPECT      check shape & feature scales
2. STANDARDIZE  StandardScaler — ALWAYS (different units wreck distance)
3. PCA          PCA(n_components=0.95) — keep variance %, NOT a magic 2
                (optional: skip if data is already low-D & clean; check with numbers)
4. CHOOSE k     sweep k, take the silhouette peak (no labels needed)
5. CLUSTER      KMeans / GMM / DBSCAN on the full working matrix
6. EVALUATE     internal: silhouette↑, davies-bouldin↓, calinski-harabasz↑
                external (if labels): ARI↑ + confusion matrix
7. PLOT         optional 2D PCA, AFTER clustering, to glance/present only
```

**Reading the metrics:**
| Metric | Range | Good | Needs labels? |
|---|---|---|---|
| Silhouette | −1 to 1 | higher (>0.5 strong) | no |
| Davies-Bouldin | 0 to ∞ | lower (~0) | no |
| Calinski-Harabasz | 0 to ∞ | higher | no |
| Inertia | 0 to ∞ | lower (compare same k only) | no |
| ARI | ~0 to 1 | higher (1=perfect) | yes |

**The mindset shift you asked about:**
- PCA uses a **variance threshold**, not a hardcoded 2. The '2' only existed for tutorial plots.
- Clustering runs on the **full working matrix** (here 10-D), never on a 2-D plot.
- Evaluation is **numbers-first** (silhouette etc.), because high-D data can't be eyeballed.
- A 2-D plot is **optional** — a cheap after-the-fact sanity check and a way to show humans, not the engine.

**Next:** kernel methods + kernel PCA (clustering curved/nonlinear shapes), then PageRank, linear regression, bootstrap.